# V6 — 00: Smoketest

빠른 sanity check: 모든 policy가 올바르게 import되고,  
single forward pass + processor pipeline이 에러 없이 돌아가는지 확인.

| Check | 내용 |
|-------|------|
| 1 | config import + factory registration |
| 2 | dataset 샘플 로드 (첫 10 프레임) |
| 3 | 각 policy 타입 forward pass (dummy batch) |
| 4 | processor pipeline (pre + post) |
| 5 | ChunkPairDataset 동작 확인 |

> GPU 필요 없음 — CPU에서 실행 가능 (dtype=float32, tiny batch).

In [ ]:
import sys, os
from pathlib import Path

# project root
PROJECT_ROOT = Path.home() / "lerobot_project" / "lerobot-ac-mlp"
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))
sys.path.insert(0, str(Path(".").resolve()))  # common_v6

DEVICE = "cpu"  # smoketest runs on CPU
B, T, D_OBS, D_ACT = 2, 100, 14, 14  # tiny batch dims

print(f"SRC_DIR: {SRC_DIR}")
print(f"SRC_DIR exists: {SRC_DIR.exists()}")

## Check 1: Import all policy configs and factory

In [ ]:
import traceback

IMPORT_CHECKS = [
    ("ACT config",             "lerobot.policies.act.configuration_act",                    "ACTConfig"),
    ("ACT policy",             "lerobot.policies.act.modeling_act",                         "ACT"),
    ("ACT-ICPE config",        "lerobot.policies.act_icpe.configuration_act_icpe",           "ACTICPEConfig"),
    ("ACT-ICPE policy",        "lerobot.policies.act_icpe.modeling_act_icpe",                "ACTICPE"),
    ("ACM config",             "lerobot.policies.acm.configuration_acm",                    "ACMConfig"),
    ("ACM policy",             "lerobot.policies.acm.modeling_acm",                         "ACM"),
    ("ACM2 config",            "lerobot.policies.acm2.configuration_acm2",                  "ACM2Config"),
    ("ACM2 policy",            "lerobot.policies.acm2.modeling_acm2",                       "ACM2"),
    ("ACM3 config",            "lerobot.policies.acm3.configuration_acm3",                  "ACM3Config"),
    ("ACM3 policy",            "lerobot.policies.acm3.modeling_acm3",                       "ACM3"),
    ("ACM3-ICPE config",       "lerobot.policies.acm3_icpe.configuration_acm3_icpe",        "ACM3ICPEConfig"),
    ("ACM3-ICPE policy",       "lerobot.policies.acm3_icpe.modeling_acm3_icpe",             "ACM3ICPE"),
    ("ACM3-SSCP config",       "lerobot.policies.acm3_sscp.configuration_acm3_sscp",        "ACM3SSCPConfig"),
    ("ACM3-SSCP policy",       "lerobot.policies.acm3_sscp.modeling_acm3_sscp",             "ACM3SSCP"),
    ("ACM3-ICPE-TSSCP config", "lerobot.policies.acm3_icpe_tsscp.configuration_acm3_icpe_tsscp", "ACM3ICPETSSCPConfig"),
    ("ACM3-ICPE-TSSCP policy", "lerobot.policies.acm3_icpe_tsscp.modeling_acm3_icpe_tsscp",      "ACM3ICPETSSCP"),
    ("factory",                "lerobot.policies.factory",                                  "make_policy"),
    ("ChunkPairDataset",       "lerobot.datasets.chunk_pair_dataset",                       "ChunkPairDataset"),
]

all_ok = True
for name, module, symbol in IMPORT_CHECKS:
    try:
        mod = __import__(module, fromlist=[symbol])
        obj = getattr(mod, symbol)
        print(f"  OK  {name:<35} ({symbol})")
    except Exception as e:
        print(f"  FAIL {name:<35} → {e}")
        all_ok = False

print()
print("Import check:", "ALL PASS" if all_ok else "SOME FAILED ← fix before training")

## Check 2: Dataset sample load

In [ ]:
try:
    from lerobot.datasets.lerobot_dataset import LeRobotDataset
    ds = LeRobotDataset("lerobot/aloha_sim_transfer_cube_human", episodes=[0])
    sample = ds[0]
    print("Dataset OK")
    print(f"  num_frames : {ds.num_frames}")
    print(f"  num_episodes: {ds.num_episodes}")
    print(f"  sample keys: {list(sample.keys())}")
    for k, v in sample.items():
        shape = getattr(v, 'shape', type(v).__name__)
        print(f"    {k}: {shape}")
except Exception as e:
    print(f"Dataset FAIL: {e}")
    import traceback; traceback.print_exc()

## Check 3: Forward pass — dummy batch on CPU

In [ ]:
import torch

# Minimal dummy batch matching AlohaTransferCube observations
# observation.images.top: (B, C, H, W), observation.state: (B, D_OBS)
# action: (B, K, D_ACT),  action_is_pad: (B, K)
K = 10  # tiny chunk size for smoketest

def make_dummy_batch(device="cpu", chunk_size=K, with_n1=False):
    b = {
        "observation.images.top":  torch.randn(B, 3, 480, 640, device=device),
        "observation.state":       torch.randn(B, 14, device=device),
        "action":                  torch.randn(B, chunk_size, 14, device=device),
        "action_is_pad":           torch.zeros(B, chunk_size, dtype=torch.bool, device=device),
    }
    if with_n1:
        b["action_n1"]        = torch.randn(B, chunk_size, 14, device=device)
        b["action_is_pad_n1"] = torch.zeros(B, chunk_size, dtype=torch.bool, device=device)
    return b

print("Dummy batch created:", {k: v.shape for k, v in make_dummy_batch().items()})

In [ ]:
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import ACT
from lerobot.policies.act_icpe.configuration_act_icpe import ACTICPEConfig
from lerobot.policies.act_icpe.modeling_act_icpe import ACTICPE
from lerobot.policies.acm.configuration_acm import ACMConfig
from lerobot.policies.acm.modeling_acm import ACM
from lerobot.policies.acm2.configuration_acm2 import ACM2Config
from lerobot.policies.acm2.modeling_acm2 import ACM2
from lerobot.policies.acm3.configuration_acm3 import ACM3Config
from lerobot.policies.acm3.modeling_acm3 import ACM3
from lerobot.policies.acm3_icpe.configuration_acm3_icpe import ACM3ICPEConfig
from lerobot.policies.acm3_icpe.modeling_acm3_icpe import ACM3ICPE
from lerobot.policies.acm3_sscp.configuration_acm3_sscp import ACM3SSCPConfig
from lerobot.policies.acm3_sscp.modeling_acm3_sscp import ACM3SSCP
from lerobot.policies.acm3_icpe_tsscp.configuration_acm3_icpe_tsscp import ACM3ICPETSSCPConfig
from lerobot.policies.acm3_icpe_tsscp.modeling_acm3_icpe_tsscp import ACM3ICPETSSCP

COMMON_FEAT = {
    "input_features": {
        "observation.images.top": {"type": "visual", "shape": [3, 480, 640]},
        "observation.state":      {"type": "low_dim", "shape": [14]},
    },
    "output_features": {
        "action": {"type": "low_dim", "shape": [14]},
    },
}

POLICY_TESTS = [
    ("ACT",           ACTConfig,           ACT,           False),
    ("ACT-ICPE",      ACTICPEConfig,       ACTICPE,       False),
    ("ACM",           ACMConfig,           ACM,           False),
    ("ACM2",          ACM2Config,          ACM2,          False),
    ("ACM3",          ACM3Config,          ACM3,          False),
    ("ACM3-ICPE",     ACM3ICPEConfig,      ACM3ICPE,      False),
    ("ACM3-SSCP",     ACM3SSCPConfig,      ACM3SSCP,      True),   # CC: with_n1
    ("ACM3-ICPE-TSSCP", ACM3ICPETSSCPConfig, ACM3ICPETSSCP, True),
]

results = []
for name, ConfigCls, ModelCls, with_n1 in POLICY_TESTS:
    try:
        cfg = ConfigCls(chunk_size=K, **COMMON_FEAT)
        cfg.device = DEVICE
        model = ModelCls(cfg).to(DEVICE).eval()
        batch = make_dummy_batch(DEVICE, K, with_n1=with_n1)
        with torch.no_grad():
            out = model.forward(batch)
        loss = out["loss"] if isinstance(out, dict) else out
        loss_val = float(loss) if torch.is_tensor(loss) else "(dict)"
        print(f"  OK  {name:<25}  loss={loss_val}")
        results.append((name, True, ""))
    except Exception as e:
        print(f"  FAIL {name:<25}  {e}")
        traceback.print_exc()
        results.append((name, False, str(e)))

print()
passed = sum(1 for _, ok, _ in results if ok)
print(f"Forward pass: {passed}/{len(results)} passed")

## Check 4: Processor pipeline

In [ ]:
from lerobot.policies.factory import make_pre_post_processors

proc_tests = [
    ("ACT",           ACTConfig(**COMMON_FEAT)),
    ("ACT-ICPE",      ACTICPEConfig(**COMMON_FEAT)),
    ("ACM3",          ACM3Config(**COMMON_FEAT)),
    ("ACM3-ICPE",     ACM3ICPEConfig(**COMMON_FEAT)),
    ("ACM3-SSCP",     ACM3SSCPConfig(**COMMON_FEAT)),
    ("ACM3-ICPE-TSSCP", ACM3ICPETSSCPConfig(**COMMON_FEAT)),
]

for name, cfg in proc_tests:
    try:
        pre, post = make_pre_post_processors(cfg, dataset_stats=None)
        print(f"  OK  {name:<25}  pre={pre.name!r}  post={post.name!r}")
    except Exception as e:
        print(f"  FAIL {name:<25}  {e}")

## Check 5: ChunkPairDataset

In [ ]:
try:
    from lerobot.datasets.chunk_pair_dataset import ChunkPairDataset

    # Re-use the dataset loaded above (episode 0 only)
    cp_ds = ChunkPairDataset(ds, chunk_size=10, dataset_stats=ds.meta.stats)
    print(f"ChunkPairDataset: {len(cp_ds)} valid pairs (chunk_size=10)")

    if len(cp_ds) > 0:
        item = cp_ds[0]
        print(f"Sample keys: {list(item.keys())}")
        assert "action_n1" in item,        "action_n1 missing"
        assert "action_is_pad_n1" in item, "action_is_pad_n1 missing"
        assert item["action_n1"].shape == item["action"].shape, "shape mismatch"
        print("  action shape:",    item["action"].shape)
        print("  action_n1 shape:", item["action_n1"].shape)
        print("ChunkPairDataset: OK")
    else:
        print("WARNING: no valid pairs (episode too short for chunk_size=10?)")
except Exception as e:
    print(f"ChunkPairDataset FAIL: {e}")
    traceback.print_exc()

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print("="*50)
print("SMOKETEST SUMMARY")
print("="*50)
print(f"  Import check    : {'PASS' if all_ok else 'FAIL'}")
print(f"  Forward pass    : {passed}/{len(results)} passed")
print()
if not all_ok or passed < len(results):
    print("  !! Fix failures above before launching full training !!")
else:
    print("  All checks passed. Ready for v6 training.")